# Phase 4C.4 — Development robustness (frozen 4C.3B models) — Colab launcher

Scores every outer-test image of the 341 development sources under six locked conditions
(`ml/configs/development_robustness_protocol.yaml`) with the **already fitted** fold models.
No refit, recalibration, training or tuning. The original condition must reproduce Phase 4C.3B
before any transformed condition runs. Primary endpoint: ΔMacro-F1 (`jpeg_q75 − original`) of
`late_fusion_stacked`; everything else is exploratory. Pilot = technical check only.

Default `MODE = "preflight"`. Outputs stay on Drive (outside Git). Zero locked-test access.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

MODE = "preflight"          # "preflight" -> "pilot" -> "full" -> "analyze"
ALLOW_FULL = False          # must be True for MODE in {"full", "analyze"}
DEVICE = "cuda"             # use the device type recorded in the ablation visual cache receipt
BRANCH = "claude/keen-knuth-4esvaw"
EXPECTED_COMMIT = ""        # paste the commit SHA from HANDOFF.md; empty = report only

ROOT = Path("/content/drive/MyDrive/Colab Notebooks/forensics-web-lab")
REPO_DIR = Path("/content/forensics-web-lab")             # a checkout of BRANCH (upload or clone)
DATASET_ARCHIVE = ROOT / "phase_4c1/inputs/phase_4c1_binary_n250_reusable.tar"
WEIGHTS = ROOT / "phase_4c1/inputs/mobilenet_v3_small-047dcff4.pth"
ABLATION_DIR = ROOT / "visual_dsp_ablation/runs/execution_ablation"      # fits/ + shared/
LATE_FUSION_DIR = ROOT / "calibrated_late_fusion/runs/execution_late_fusion"
OUTPUT_DIR = ROOT / "development_robustness/runs/execution_robustness"
REPORT_DIR = ROOT / "development_robustness/report"
BUNDLE = Path("/content/development_bundle")


In [ ]:
import subprocess, sys, tarfile, shutil
from pathlib import PurePosixPath

assert MODE in {"preflight", "pilot", "full", "analyze"}
assert MODE not in {"full", "analyze"} or ALLOW_FULL, "full/analyze requires ALLOW_FULL=True"
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch",
                    "https://github.com/nomozer/forensics-web-lab", str(REPO_DIR)], check=True)
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("repository HEAD:", head)
if EXPECTED_COMMIT:
    assert head == EXPECTED_COMMIT, "Checkout does not match the commit in HANDOFF.md"
for path in (DATASET_ARCHIVE, ABLATION_DIR, LATE_FUSION_DIR):
    assert path.exists(), f"Missing input: {path}"
if not (BUNDLE / "manifest_pilot_a_option_p.csv").is_file():
    BUNDLE.mkdir(parents=True, exist_ok=True)
    with tarfile.open(DATASET_ARCHIVE, "r:") as archive:
        for member in archive.getmembers():
            posix = PurePosixPath(member.name)
            assert (member.isfile() or member.isdir()) and not posix.is_absolute() and ".." not in posix.parts
        archive.extractall(BUNDLE)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
common = ["--manifest", str(BUNDLE / "manifest_pilot_a_option_p.csv"), "--data-root", str(BUNDLE),
          "--weights-path", str(WEIGHTS), "--late-fusion-dir", str(LATE_FUSION_DIR),
          "--ablation-dir", str(ABLATION_DIR), "--output-dir", str(OUTPUT_DIR), "--device", DEVICE]
if MODE == "analyze":
    cmd = [sys.executable, "scripts/research/analyze_development_robustness.py",
           "--output-dir", str(OUTPUT_DIR), "--report-dir", str(REPORT_DIR)]
else:
    cmd = [sys.executable, "-m", "ml.training.run_development_robustness", "--mode", MODE, *common]
print("Executing:", " ".join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)
if MODE == "analyze":
    print((REPORT_DIR / "ROBUSTNESS_REPORT.md").read_text())
